# Notebook Description

This notebook performs ETL from bronze to silver for PHHousing property data. It reads raw housing listings, filters for residential properties with price >= 10,000, enriches missing city/province info using PSGC reference tables and listing titles, and outputs a cleaned Delta table.

**Input:**  
- Bronze parquet: `housing_raw`
- Bronze Delta: `psgc_cities`, `psgc_provinces`

**Output:**  
- Silver Delta: `dim_housing` (residential properties with city/province info)

**Schema:**
- id string
- sourceSlug: string
- sourceName: string
- title: string
- price: decimal(18,2)
- priceFormatted: string
- pricePerSqm: integer
- floorArea: decimal(18,2)
- lotArea: decimal(18,2)
- city: string
- province: string
- isNew: string
- daysListed: integer
- listingScore: decimal(18,2)
- firstSeenAt: string
- geographyFk: string 
 

In [0]:
bronze_folder = 'abfss://bronze@asterisktotle01.dfs.core.windows.net/PHHousing'
silver_folder = 'abfss://silver@asterisktotle01.dfs.core.windows.net/PHHousing'




In [0]:
# Cleaning

from pyspark.sql import functions as F

housing_raw = spark.read.format("parquet").load(f"{bronze_folder}/housing_raw/housing_raw.parquet")

# Filter the housing price with price less than 
housing_filtered = housing_raw.filter(
    (F.col("propertyType") == "residential") &
    (F.col("price") >= 10000 )
)

housing_filtered = housing_filtered.select(
    "id",
    "sourceSlug",
    "sourceName",
    "title",
    "price",
    "priceFormatted",
    "pricePerSqm",
    "floorArea",
    "lotArea",
    "city",
    "province",
    "isNew",
    "daysListed",
    "listingScore",
    "firstSeenAt"
)
print(f'Number of residential property: {housing_filtered.count()}')


In [0]:
# Rows that already have location (city and province) — leave untouched
housing_has_location = housing_filtered.filter(
    ((F.col('city').isNotNull()) & (F.col('city') != '')) |
    ((F.col('province').isNotNull()) & (F.col('province') != ''))
)

print("Number of residential property with location (city/province): ", housing_has_location.count())


In [0]:
import re
city_df = spark.read.format('delta').load(bronze_folder + '/psgc_cities')
province_df = spark.read.format('delta').load(bronze_folder + '/psgc_provinces')

city_list = []

city_table = city_df.select('name').collect()

for row in city_table:
    city_name = row['name'].strip()
    clean_name = re.sub(r'^(City of|Science City of|Island Garden City of)\s+', '', city_name, flags=re.IGNORECASE).strip()
    
    city_list.append(clean_name)
# display(city_list)

In [0]:
housing_with_no_location = housing_filtered.filter(
    ((F.col('city').isNull()) | (F.col('city') == '')) &
    ((F.col('province').isNull()) | (F.col('province') == '')))
print('Number of housing with no location (city/province): ', housing_with_no_location.count())

In [0]:
import re

city_list_sorted = sorted(city_list, key=len, reverse=True)
city_pattern = "|".join(re.escape(c) for c in city_list_sorted)

# search for the 'city' in the 'title' and write it on 'cityFromTitle'
housing_enriched = housing_with_no_location.withColumn(
    "cityFromTitle",
    F.regexp_extract(F.col("title"), f"(?i)({city_pattern})", 1)
)

# Write the nul value if 'cityFromTitle' is null else use its value
housing_enriched = housing_enriched.withColumn(
    "cityFromTitle",
    F.when(F.col("cityFromTitle") == "", F.lit(None)).otherwise(F.col("cityFromTitle"))
)
# If column 'city' city has value, write the city else use the value from 'cityFromTitle'
housing_enriched_city = housing_enriched.withColumn(
    "city",
    F.coalesce(F.col("city"), F.col("cityFromTitle"))
)




In [0]:
province_df = spark.read.format('delta').load(bronze_folder + '/psgc_provinces')
provinces_distinct = province_df.select('name').distinct()
province_list = []

for row in provinces_distinct.collect():
    province_name = row["name"]
    if province_name is not None and province_name != "":
        province_list.append(province_name)
print("Number of provincess: ", len(province_list))


In [0]:
import re 

province_list_sorted = sorted(province_list,key=len, reverse=True)
province_pattern = "|".join(re.escape(p) for p in province_list_sorted)

housing_enriched_province = housing_enriched_city.withColumn(
    "provinceFromTitle",
    F.regexp_extract(F.col("title"), f"(?i)({province_pattern})", 1)
)
# Clean up the empty strings to true Nulls
housing_enriched_province = housing_enriched_province.withColumn(
    "provinceFromTitle",
    F.when(F.col("provinceFromTitle") == "", F.lit(None)).otherwise(F.col("provinceFromTitle"))
)

housing_enriched_city_province = housing_enriched_province.withColumn(
    "province",
    F.coalesce(F.col("province"), F.col("provinceFromTitle"))
)

housing_with_new_location = housing_enriched_city_province.filter(
    (F.col('province').isNotNull()) & (F.col('province') != '') | 
    (F.col('city').isNotNull() & (F.col('city') != ''))
)

print('Number of housing with new location (city or province): ' , housing_with_new_location.count() )



In [0]:

# Join the two Dataframes (housing with location and new location)
housing_silver = housing_has_location.unionByName(housing_with_new_location, allowMissingColumns = True)

# Normalize 'Ncr' as 'Metro Manila'
housing_silver = housing_silver.withColumn(
    "province", 
    F.when(F.col("province").rlike("(?i)^Ncr"), F.lit("Metro Manila"))
    .otherwise(F.col("province"))
    )
# Flag the the 'Mindoro' as null
housing_silver = housing_silver.withColumn(
    "province",
    F.when(F.col("province") == "Mindoro", F.lit(None))
    .otherwise(F.col("province"))
)

#lower case the province and city name
housing_silver = (housing_silver.
                  withColumn("province",
                    F.lower(F.col("province"))                           
                )).withColumn("city", F.lower(F.col("city"))).drop('provinceFromTitle')
                



In [0]:


# remove "(anything inside parenthesis)"
housing_silver = housing_silver.withColumn(
    "cityClean",
    F.regexp_replace(F.col("city"), r"\s*\([^)]*\)", "")
)

# remove trailing comma/period noise and collapse extra spaces
housing_silver = housing_silver.withColumn(
    "cityClean",
    F.trim(F.regexp_replace(F.col("cityClean"), r"[,\.]+$", ""))
)

# strip "City of" / "Municipality of" prefix
housing_silver = housing_silver.withColumn(
    "cityClean",
    F.regexp_replace(F.col("cityClean"), r"(?i)^(city of|municipality of)\s+", "")
)

# strip trailing " City" suffix
housing_silver = housing_silver.withColumn(
    "cityClean",
    F.regexp_replace(F.col("cityClean"), r"(?i)\s+city$", "")
)

# remove "i/ii" type suffixes (e.g. Tondo I/II)
housing_silver = housing_silver.withColumn(
    "cityClean",
    F.trim(F.regexp_replace(F.col("cityClean"), r"(?i)\s+i/ii$", ""))
)

# transform "sta." to "santa" and "sto." to "santo"
housing_silver = housing_silver.withColumn(
    "cityClean",
    F.when(F.col("cityClean").rlike(r"(?i)^Sta\."), F.regexp_replace(F.col("cityClean"), r"(?i)^Sta\.", "santa"))
    .when(F.col("cityClean").rlike(r"(?i)Sto.\\."), F.regexp_replace(F.col("cityClean"), r"(?i)^Sto.\.", "santo"))
    .otherwise(F.col("cityClean"))
)

# Drop the temporary column "cityClean" and 'cityFromTitle'
housing_silver = (
    housing_silver
    .withColumn("city", F.col("cityClean"))
    .drop("cityClean")
    .drop('cityFromTitle')
)

In [0]:
print("housing with exisiting location: ",housing_has_location.count())
print("housing with new location: ",housing_with_new_location.count())
print("combined toal: ",housing_silver.count())
print(housing_silver.printSchema())

In [0]:
dim_geography = spark.read.format("delta").load(silver_folder + '/dim_geography')

# Create a foreign key for the city and province

housing_fk = (housing_silver.
              join(dim_geography, 
                        (F.col('city') == F.col('geographyName')) |
                        (F.col('province') == F.col('geographyName')),
                        'left')
              .withColumn('geographyFk', F.col('geographyFk'))
              .drop('cityCode', 'provinceCode', 'cityName', 'provinceName', 'islandGroupCode',
                    'propertyType')
              )

housing_fk.write.format("delta").mode("overwrite").save(silver_folder + '/dim_housing')
print("dim_housing table created")
